# Market Data Pre-Download

Use this notebook before running the simulation notebook.
It downloads each ticker in a specified date range and saves one CSV per ticker in `data/` with the same format used by the project (`Date,Adj Close`).

In [1]:
from __future__ import annotations

from pathlib import Path
import time
import pandas as pd

from data_loader import download_adj_close_prices

In [ ]:
# Configure your download window and tickers here
# Full ticker universe from the benchmark table (duplicates removed)
TICKERS = [
    "CSSPX.MI",
    "SWDA.MI",
    "EIMI.MI",
    "CSNDX.MI",
    "IUSQ.MI",
    "VWCE.MI",
    "EXS1.MI",
    "MEUD.MI",
    "SMEA.MI",
    "XD9U.MI",
    "XMME.MI",
    "CSSX5E.MI",
    "EMUL.MI",
    "XLRE",
    "GOVT",
    "TLH",
    "LTPZ",
    "XTLT.TO",
    "UTHY",
    "TIP",
    "IGLA.L",
    "XG7S.DE",
    "SEGA.MI",
    "VAGF.DE",
    "BTC=F",
    "GC=F",
    "SI=F",
    "CL=F",
    "CSH.PA",
    "PDBC",
    "VGLT",
]
START_DATE = "2010-04-03"
END_DATE = "2026-04-14"

# Small delay between requests helps reduce provider throttling
COOLDOWN_SECONDS = 4

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Tickers: {TICKERS}")
print(f"Window: {START_DATE} -> {END_DATE}")
print(f"Output dir: {DATA_DIR.resolve()}")

Tickers: ['CSSPX.MI', 'SWDA.MI', 'EIMI.MI', 'CSNDX.MI', 'IUSQ.MI', 'VWCE.MI', 'EXS1.MI', 'MEUD.MI', 'SMEA.MI', 'XD9U.MI', 'XMME.MI', 'CSSX5E.MI', 'EMUL.MI', 'XLRE', 'GOVT', 'TLH', 'LTPZ', 'XTLT.TO', 'UTHY', 'TIP', 'IGLA.L', 'XG7S.DE', 'SEGA.MI', 'VAGF.DE', 'BTC=F', 'GC=F', 'SI=F', 'CL=F', 'CSH.PA', 'PDBC', 'VGLT']
Window: 2010-04-03 -> 2026-04-14
Output dir: /Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data


In [3]:
results = []

for i, ticker in enumerate(TICKERS, start=1):
    print(f"[{i}/{len(TICKERS)}] Downloading {ticker}...")
    status = "ok"
    rows = 0
    error = ""

    out_path = DATA_DIR / f"{ticker}.csv"

    try:
        req_start = pd.Timestamp(START_DATE)
        req_end = pd.Timestamp(END_DATE)
        one_day = pd.Timedelta(days=1)

        existing_series = None
        local_min = None
        local_max = None

        if out_path.exists():
            existing_df = pd.read_csv(out_path, parse_dates=["Date"])
            if not existing_df.empty and {"Date", "Adj Close"}.issubset(existing_df.columns):
                existing_series = (
                    existing_df.set_index("Date")["Adj Close"]
                    .astype(float)
                    .dropna()
                    .sort_index()
                )
                if not existing_series.empty:
                    local_min = existing_series.index.min()
                    local_max = existing_series.index.max()

        needs_left = local_min is None or req_start < local_min
        needs_right = local_max is None or req_end > local_max

        if not needs_left and not needs_right:
            # Requested window is fully covered locally: keep file untouched.
            final_series = existing_series
            print("    local data already covers requested window; no download needed")
        else:
            new_parts = []

            if needs_left:
                left_end = (local_min - one_day) if local_min is not None else req_end
                if req_start <= left_end:
                    left_prices = download_adj_close_prices(
                        tickers=[ticker],
                        start=req_start.strftime("%Y-%m-%d"),
                        end=left_end.strftime("%Y-%m-%d"),
                    )
                    left_series = left_prices[ticker].dropna()
                    new_parts.append(left_series)

            if needs_right:
                right_start = (local_max + one_day) if local_max is not None else req_start
                if right_start <= req_end:
                    right_prices = download_adj_close_prices(
                        tickers=[ticker],
                        start=right_start.strftime("%Y-%m-%d"),
                        end=req_end.strftime("%Y-%m-%d"),
                    )
                    right_series = right_prices[ticker].dropna()
                    new_parts.append(right_series)

            pieces = []
            if existing_series is not None and not existing_series.empty:
                pieces.append(existing_series)
            pieces.extend(new_parts)

            if not pieces:
                raise RuntimeError("No local data and no downloaded data available.")

            final_series = pd.concat(pieces).sort_index()
            final_series = final_series[~final_series.index.duplicated(keep="last")]

        # Force project-consistent on-disk shape: Date index + Adj Close column.
        out_df = final_series.to_frame(name="Adj Close")
        out_df.index.name = "Date"
        out_df.to_csv(out_path, date_format="%Y-%m-%d")

        rows = int(out_df.shape[0])
        print(f"    saved {rows} rows to {out_path}")
    except Exception as exc:
        status = "failed"
        error = str(exc)
        print(f"    FAILED: {ticker}: {error}")

    results.append({"ticker": ticker, "status": status, "rows": rows, "error": error})

    if i < len(TICKERS):
        time.sleep(COOLDOWN_SECONDS)

summary = pd.DataFrame(results)
summary

[1/31] Downloading CSSPX.MI...


/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:265: RuntimeWarning: Local cache miss for symbols: ['CSSPX.MI']. Attempting online providers.
  warnings.warn(

1 Failed download:
['CSSPX.MI']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:318: RuntimeWarning: Batch yfinance download returned incomplete data. Falling back to per-ticker recovery for: ['CSSPX.MI']
  warnings.warn(
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:327: RuntimeWarning: Yahoo Finance batch response returned no usable rows for all uncached symbols. Likely temporary provider throttling; skipping repeated per-ticker Yahoo calls.
  warnings.warn(
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:349: RuntimeWarning: No fallback data source could recover CSSPX.MI after Yahoo batch failure. Try again later or provide a local cache CSV in data/.
  warnings.war

    FAILED: CSSPX.MI: No price data returned by yfinance for requested inputs.
[2/31] Downloading SWDA.MI...


/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:265: RuntimeWarning: Local cache miss for symbols: ['SWDA.MI']. Attempting online providers.
  warnings.warn(

1 Failed download:
['SWDA.MI']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:318: RuntimeWarning: Batch yfinance download returned incomplete data. Falling back to per-ticker recovery for: ['SWDA.MI']
  warnings.warn(
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:327: RuntimeWarning: Yahoo Finance batch response returned no usable rows for all uncached symbols. Likely temporary provider throttling; skipping repeated per-ticker Yahoo calls.
  warnings.warn(
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:349: RuntimeWarning: No fallback data source could recover SWDA.MI after Yahoo batch failure. Try again later or provide a local cache CSV in data/.
  warnings.warn(


    FAILED: SWDA.MI: No price data returned by yfinance for requested inputs.
[3/31] Downloading EIMI.MI...


...

    FAILED: PDBC: No price data returned by yfinance for requested inputs.
[31/31] Downloading VGLT...


/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:265: RuntimeWarning: Local cache miss for symbols: ['VGLT']. Attempting online providers.
  warnings.warn(

1 Failed download:
['VGLT']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:318: RuntimeWarning: Batch yfinance download returned incomplete data. Falling back to per-ticker recovery for: ['VGLT']
  warnings.warn(
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:327: RuntimeWarning: Yahoo Finance batch response returned no usable rows for all uncached symbols. Likely temporary provider throttling; skipping repeated per-ticker Yahoo calls.
  warnings.warn(


    FAILED: VGLT: No price data returned by yfinance for requested inputs.


/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:349: RuntimeWarning: No fallback data source could recover VGLT after Yahoo batch failure. Try again later or provide a local cache CSV in data/.
  warnings.warn(


,ticker,status,rows,error
0,CSSPX.MI,failed,0,No price data returned by yfinance for request...
1,SWDA.MI,failed,0,No price data returned by yfinance for request...
2,EIMI.MI,failed,0,No price data returned by yfinance for request...
3,CSNDX.MI,failed,0,No price data returned by yfinance for request...
4,IUSQ.MI,failed,0,No price data returned by yfinance for request...
5,VWCE.MI,failed,0,No price data returned by yfinance for request...
6,EXS1.MI,failed,0,No price data returned by yfinance for request...
7,MEUD.MI,failed,0,No price data returned by yfinance for request...
8,SMEA.MI,failed,0,No price data returned by yfinance for request...
9,XD9U.MI,failed,0,No price data returned by yfinance for request...


In [4]:
failed = summary[summary["status"] == "failed"]
if failed.empty:
    print("All ticker files downloaded successfully.")
else:
    print("Some tickers failed. Re-run only failed tickers by replacing TICKERS with:")
    print(failed["ticker"].tolist())

Some tickers failed. Re-run only failed tickers by replacing TICKERS with:
['CSSPX.MI', 'SWDA.MI', 'EIMI.MI', 'CSNDX.MI', 'IUSQ.MI', 'VWCE.MI', 'EXS1.MI', 'MEUD.MI', 'SMEA.MI', 'XD9U.MI', 'XMME.MI', 'CSSX5E.MI', 'EMUL.MI', 'XLRE', 'GOVT', 'TLH', 'LTPZ', 'XTLT.TO', 'UTHY', 'TIP', 'IGLA.L', 'XG7S.DE', 'SEGA.MI', 'VAGF.DE', 'CSH.PA', 'PDBC', 'VGLT']
